In [1]:
import itertools
import math
from typing import Dict, List, Tuple, Union, Iterable, Optional

import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:

# ============================================================
# 1) SYNTHETIC DATA GENERATION
# ============================================================

def generate_synthetic_domain_data(
    n_rows: int = 800,
    seed: int = 42,
    domain_feature_counts: Optional[Dict[str, int]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Generate synthetic pandas DataFrame with 8 domains and configurable
    feature counts per domain (1 to 8 each).

    Returns
    -------
    df : pd.DataFrame
        Synthetic feature dataframe + target column.
    feature_map : pd.DataFrame
        Mapping table with columns: domain, feature_name
    """
    rng = np.random.default_rng(seed)

    if domain_feature_counts is None:
        domain_feature_counts = {
            "pricing": 5,
            "macro": 4,
            "promo": 3,
            "menu": 6,
            "media": 4,
            "operations": 5,
            "csat": 3,
            "weather": 2,
        }

    # validate
    if len(domain_feature_counts) != 8:
        raise ValueError("Please provide exactly 8 domains in domain_feature_counts.")
    for d, c in domain_feature_counts.items():
        if not (1 <= c <= 8):
            raise ValueError(f"Domain '{d}' must have between 1 and 8 features. Got: {c}")

    # shared latent factors to create cross-domain correlations
    global_latent_1 = rng.normal(0, 1, n_rows)
    global_latent_2 = rng.normal(0, 1, n_rows)

    data = {}
    feature_map_rows = []
    all_feature_names = []
    feature_counter = 1

    # Each domain gets 2-3 latent factors depending on feature count
    for domain, n_features in domain_feature_counts.items():
        n_domain_latents = min(3, max(1, math.ceil(n_features / 3)))
        domain_latents = [rng.normal(0, 1, n_rows) for _ in range(n_domain_latents)]

        for j in range(n_features):
            feature_name = f"feature{feature_counter}"
            feature_counter += 1

            # assign main domain latent in a clustered way
            main_latent_idx = j % n_domain_latents
            main_loading = rng.uniform(0.75, 0.95)

            # light secondary loading for some features to create partial overlap
            secondary_loading = rng.uniform(0.10, 0.30) if n_domain_latents > 1 else 0.0
            secondary_latent_idx = (main_latent_idx + 1) % n_domain_latents if n_domain_latents > 1 else main_latent_idx

            # shared/global correlation loadings
            global_loading_1 = rng.uniform(0.00, 0.20)
            global_loading_2 = rng.uniform(0.00, 0.20)

            noise_scale = rng.uniform(0.25, 0.45)

            x = (
                main_loading * domain_latents[main_latent_idx]
                + secondary_loading * domain_latents[secondary_latent_idx]
                + global_loading_1 * global_latent_1
                + global_loading_2 * global_latent_2
                + rng.normal(0, noise_scale, n_rows)
            )

            data[feature_name] = x
            feature_map_rows.append({"domain": domain, "feature_name": feature_name})
            all_feature_names.append(feature_name)

    df = pd.DataFrame(data)

    # synthetic target: combine a handful of true drivers
    # choose every ~4th feature as a "true" driver for realism
    chosen_features = all_feature_names[::4][: min(10, len(all_feature_names[::4]))]
    coefs = rng.uniform(-1.5, 1.5, len(chosen_features))

    y = np.zeros(n_rows)
    for coef, feat in zip(coefs, chosen_features):
        y += coef * df[feat].values

    y += rng.normal(0, 1.25, n_rows)
    df["target"] = y

    feature_map = pd.DataFrame(feature_map_rows)
    return df, feature_map


In [5]:
# ============================================================
# 2) HELPER FUNCTIONS
# ============================================================

def resolve_allowed_sizes(
    spec: Union[int, Tuple[int, int], List[int], set],
    n_features_available: int
) -> List[int]:
    """
    Convert user input into a sorted list of allowed subset sizes.

    Supported inputs:
    - int: exact subset size, e.g. 1
    - tuple(min_size, max_size): e.g. (1, 3)
    - list/set of explicit sizes: e.g. [1, 2, 4]
    """
    if isinstance(spec, int):
        sizes = [spec]
    elif isinstance(spec, tuple) and len(spec) == 2:
        min_size, max_size = spec
        sizes = list(range(min_size, max_size + 1))
    elif isinstance(spec, (list, set)):
        sizes = sorted(list(spec))
    else:
        raise ValueError(f"Unsupported restriction spec: {spec}")

    sizes = sorted(set([s for s in sizes if 1 <= s <= n_features_available]))
    return sizes


def get_domain_to_features(feature_map: pd.DataFrame) -> Dict[str, List[str]]:
    return (
        feature_map.groupby("domain")["feature_name"]
        .apply(list)
        .to_dict()
    )


def compute_abs_corr_stats(df: pd.DataFrame, features: List[str]) -> Dict[str, float]:
    """
    Compute pairwise absolute correlation summary.
    """
    if len(features) <= 1:
        return {
            "max_abs_corr": 0.0,
            "mean_abs_corr": 0.0,
            "n_corr_pairs": 0
        }

    corr = df[features].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    vals = upper.stack().values
    if len(vals) == 0:
        return {
            "max_abs_corr": 0.0,
            "mean_abs_corr": 0.0,
            "n_corr_pairs": 0
        }

    return {
        "max_abs_corr": float(np.max(vals)),
        "mean_abs_corr": float(np.mean(vals)),
        "n_corr_pairs": int(len(vals))
    }


def compute_vif_table(df: pd.DataFrame, features: List[str]) -> pd.DataFrame:
    """
    Compute VIF for a list of features.
    """
    if len(features) == 1:
        return pd.DataFrame({
            "feature_name": features,
            "vif": [1.0]
        })

    X = df[features].copy()

    # If any column is constant, VIF can fail. Drop zero-variance columns check.
    nunique = X.nunique(dropna=False)
    zero_var_cols = nunique[nunique <= 1].index.tolist()
    if zero_var_cols:
        raise ValueError(f"Zero variance features found: {zero_var_cols}")

    # statsmodels VIF expects numeric numpy matrix
    X_np = X.astype(float).values

    vifs = []
    for i, col in enumerate(features):
        vif_val = variance_inflation_factor(X_np, i)
        vifs.append({"feature_name": col, "vif": float(vif_val)})

    return pd.DataFrame(vifs)


def evaluate_feature_subset(
    df: pd.DataFrame,
    features: List[str],
    corr_threshold: float = 0.70,
    vif_threshold: float = 5.0
) -> Dict:
    """
    Evaluate one subset of features for pairwise correlation and VIF.
    """
    corr_stats = compute_abs_corr_stats(df, features)
    vif_table = compute_vif_table(df, features)
    max_vif = float(vif_table["vif"].max())
    mean_vif = float(vif_table["vif"].mean())

    passed_corr = corr_stats["max_abs_corr"] <= corr_threshold
    passed_vif = max_vif <= vif_threshold
    passed_all = passed_corr and passed_vif

    # ranking score:
    # prefer more features, lower corr, lower vif
    # this can be adjusted later
    score = (
        5.0 * len(features)
        - 3.0 * corr_stats["max_abs_corr"]
        - 1.0 * corr_stats["mean_abs_corr"]
        - 0.8 * max_vif
        - 0.2 * mean_vif
    )

    return {
        "features": list(features),
        "n_features": len(features),
        "max_abs_corr": corr_stats["max_abs_corr"],
        "mean_abs_corr": corr_stats["mean_abs_corr"],
        "max_vif": max_vif,
        "mean_vif": mean_vif,
        "passed_corr": passed_corr,
        "passed_vif": passed_vif,
        "passed_all": passed_all,
        "score": score,
        "vif_table": vif_table
    }


def options_to_dataframe(options: List[Dict], domain: Optional[str] = None) -> pd.DataFrame:
    """
    Convert list of subset option dicts to a clean DataFrame.
    """
    if not options:
        cols = ["domain", "features", "n_features", "max_abs_corr", "mean_abs_corr",
                "max_vif", "mean_vif", "passed_corr", "passed_vif", "passed_all", "score"]
        return pd.DataFrame(columns=cols)

    rows = []
    for opt in options:
        rows.append({
            "domain": domain,
            "features": tuple(opt["features"]),
            "n_features": opt["n_features"],
            "max_abs_corr": opt["max_abs_corr"],
            "mean_abs_corr": opt["mean_abs_corr"],
            "max_vif": opt["max_vif"],
            "mean_vif": opt["mean_vif"],
            "passed_corr": opt["passed_corr"],
            "passed_vif": opt["passed_vif"],
            "passed_all": opt["passed_all"],
            "score": opt["score"],
        })
    return pd.DataFrame(rows).sort_values(
        ["score", "n_features", "max_vif", "max_abs_corr"],
        ascending=[False, False, True, True]
    ).reset_index(drop=True)



In [31]:
# ============================================================
# 3) STAGE 1: WITHIN-DOMAIN SUBSET OPTIONS
# ============================================================

def generate_within_domain_options(
    df: pd.DataFrame,
    feature_map: pd.DataFrame,
    domain_feature_restrictions: Dict[str, Union[int, Tuple[int, int], List[int], set]],
    corr_threshold: float = 0.70,
    vif_threshold: float = 5.0,
    top_k_per_domain: Optional[int] = 10,
) -> Dict[str, Dict[str, pd.DataFrame]]:
    """
    For each domain:
    - enumerate subsets under the domain-specific size restriction
    - evaluate corr and VIF
    - keep valid options
    - return all valid options and top-k

    Returns
    -------
    dict like:
    {
      "pricing": {
         "all_options": DataFrame,
         "top_k_options": DataFrame
      },
      ...
    }
    """
    domain_to_features = get_domain_to_features(feature_map)
    results = {}

    for domain, features in domain_to_features.items():
        if domain not in domain_feature_restrictions:
            raise ValueError(f"Missing restriction for domain: {domain}")

        allowed_sizes = resolve_allowed_sizes(
            domain_feature_restrictions[domain],
            n_features_available=len(features)
        )

        valid_options = []

        for size in allowed_sizes:
            for combo in itertools.combinations(features, size):
                eval_result = evaluate_feature_subset(
                    df=df,
                    features=list(combo),
                    corr_threshold=corr_threshold,
                    vif_threshold=vif_threshold
                )
                if eval_result["passed_all"]:
                    valid_options.append(eval_result)

        # rank
        valid_options = sorted(
            valid_options,
            key=lambda x: (x["score"], x["n_features"], -x["max_vif"], -x["max_abs_corr"]),
            reverse=True
        )

        all_df = options_to_dataframe(valid_options, domain=domain)
        top_df = all_df.head(top_k_per_domain).copy() if top_k_per_domain is not None else all_df.copy()

        results[domain] = {
            "all_options": all_df,
            "top_k_options": top_df
        }

    return results


# ============================================================
# 4) STAGE 2: ACROSS-DOMAIN COMBINATIONS
# ============================================================

def build_option_lookup_from_stage1(
    stage1_results: Dict[str, Dict[str, pd.DataFrame]],
    use_top_k_only: bool = True
) -> Dict[str, List[List[str]]]:
    """
    Convert stage1 results into:
    {
      domain: [ [featA, featB], [featC], ... ]
    }
    """
    out = {}
    for domain, content in stage1_results.items():
        df_opts = content["top_k_options"] if use_top_k_only else content["all_options"]
        combos = [list(x) for x in df_opts["features"].tolist()]
        out[domain] = combos
    return out


def combine_domain_options(
    df: pd.DataFrame,
    stage1_results: Dict[str, Dict[str, pd.DataFrame]],
    corr_threshold: float = 0.70,
    vif_threshold: float = 5.0,
    use_top_k_only_from_stage1: bool = True,
    top_k_global: Optional[int] = 50,
    max_global_combinations: int = 200000,
    max_checked: Optional[int] = None,
) -> Dict[str, pd.DataFrame]:
    """
    Combine one valid option from each domain, then check global corr/VIF.

    Returns
    -------
    {
      "all_global_options": DataFrame,
      "top_k_global_options": DataFrame
    }
    """
    domain_option_lookup = build_option_lookup_from_stage1(
        stage1_results,
        use_top_k_only=use_top_k_only_from_stage1
    )

    domains = list(domain_option_lookup.keys())
    domain_option_lists = [domain_option_lookup[d] for d in domains]

    # guard against empty domains
    empty_domains = [d for d, opts in domain_option_lookup.items() if len(opts) == 0]
    if empty_domains:
        raise ValueError(
            f"These domains have no valid stage1 options under current thresholds/restrictions: {empty_domains}"
        )

    total_combinations = 1
    for opts in domain_option_lists:
        total_combinations *= len(opts)

    print(f"Potential global combinations to test: {total_combinations:,}")

    if total_combinations > max_global_combinations:
        raise ValueError(
            f"Too many global combinations ({total_combinations:,}). "
            f"Reduce top_k_per_domain, tighten restrictions, or increase max_global_combinations."
        )

    valid_global_rows = []
    checked = 0

    for product_tuple in itertools.product(*domain_option_lists):
        checked += 1

        # Optional early stop based on number of combinations evaluated
        if (max_checked is not None) and (checked > max_checked):
            break

        combined_features = []
        domain_option_dict = {}

        for domain, subset in zip(domains, product_tuple):
            domain_option_dict[domain] = subset
            combined_features.extend(subset)

        eval_result = evaluate_feature_subset(
            df=df,
            features=combined_features,
            corr_threshold=corr_threshold,
            vif_threshold=vif_threshold
        )

        if eval_result["passed_all"]:
            valid_global_rows.append({
                "domain_subsets": domain_option_dict,
                "features": tuple(combined_features),
                "n_features": eval_result["n_features"],
                "max_abs_corr": eval_result["max_abs_corr"],
                "mean_abs_corr": eval_result["mean_abs_corr"],
                "max_vif": eval_result["max_vif"],
                "mean_vif": eval_result["mean_vif"],
                "score": eval_result["score"]
            })

    all_global_df = pd.DataFrame(valid_global_rows)

    if all_global_df.empty:
        top_global_df = all_global_df.copy()
    else:
        all_global_df = all_global_df.sort_values(
            ["score", "n_features", "max_vif", "max_abs_corr"],
            ascending=[False, False, True, True]
        ).reset_index(drop=True)

        top_global_df = (
            all_global_df.head(top_k_global).copy()
            if top_k_global is not None
            else all_global_df.copy()
        )

    print(f"Checked global combinations: {checked:,}")
    print(f"Valid global combinations: {len(all_global_df):,}")

    return {
        "all_global_options": all_global_df,
        "top_k_global_options": top_global_df
    }



In [8]:
df, feature_map = generate_synthetic_domain_data(
        n_rows=1000,
        seed=42,
        domain_feature_counts={
            "pricing": 5,
            "macro": 4,
            "promo": 2,
            "menu": 6,
            "media": 4,
            "operations": 5,
            "csat": 3,
            "weather": 2,
        }
    )

print("\nFeature map:")
print(feature_map.head(20))

print("\nSynthetic dataframe shape:")
print(df.shape)



Feature map:
     domain feature_name
0   pricing     feature1
1   pricing     feature2
2   pricing     feature3
3   pricing     feature4
4   pricing     feature5
5     macro     feature6
6     macro     feature7
7     macro     feature8
8     macro     feature9
9     promo    feature10
10    promo    feature11
11     menu    feature12
12     menu    feature13
13     menu    feature14
14     menu    feature15
15     menu    feature16
16     menu    feature17
17    media    feature18
18    media    feature19
19    media    feature20

Synthetic dataframe shape:
(1000, 32)


In [9]:
# --------------------------------------------------------
    # Domain-specific size restrictions
    # Example:
    # - pricing allows 1 to 3 features
    # - macro allows 1 to 2 features
    # - promo allows exactly 1 feature
    # - menu allows 1 to 3 features
    # - media allows 1 to 2 features
    # - operations allows 1 to 3 features
    # - csat allows 1 feature
    # - weather allows 1 to 2 features
    # --------------------------------------------------------
domain_feature_restrictions = {
        "pricing": (1, 3),
        "macro": (1, 2),
        "promo": 1,
        "menu": (1, 3),
        "media": (1, 2),
        "operations": (1, 3),
        "csat": 1,
        "weather": (1, 2),
    }


In [26]:
# --------------------------------------------------------
# Stage 1: Within-domain valid subset options
# --------------------------------------------------------
stage1_results = generate_within_domain_options(
        df=df.drop(columns=["target"]),   # only X features for screening
        feature_map=feature_map,
        domain_feature_restrictions=domain_feature_restrictions,
        corr_threshold=0.30,
        vif_threshold=5.0,
        top_k_per_domain=8
    )

print("\n========== STAGE 1: WITHIN-DOMAIN TOP OPTIONS ==========")
combined_stage1_frames = []
for domain, res in stage1_results.items():
        print(f"\n--- Domain: {domain} ---")
        print("Top options:")
        top_options = res["top_k_options"].head(10)
        print(top_options)
        if not top_options.empty:
            combined_stage1_frames.append(top_options)

if combined_stage1_frames:
        combined_stage1_results = pd.concat(combined_stage1_frames, ignore_index=True)
        combined_stage1_results = combined_stage1_results.sort_values(
            ["score", "n_features", "max_vif", "max_abs_corr"],
            ascending=[False, False, True, True]
        ).reset_index(drop=True)

        print("\n========== STAGE 1: COMBINED TOP OPTIONS ACROSS ALL DOMAINS ==========")
        print(combined_stage1_results)
else:
        combined_stage1_results = pd.DataFrame()
        print("\nNo valid Stage 1 options found across domains.")


========== STAGE 1: WITHIN-DOMAIN TOP OPTIONS ==========

--- Domain: csat ---
Top options:
  domain      features  n_features  max_abs_corr  mean_abs_corr  max_vif  \
0   csat  (feature27,)           1           0.0            0.0      1.0   
1   csat  (feature28,)           1           0.0            0.0      1.0   
2   csat  (feature29,)           1           0.0            0.0      1.0   

   mean_vif  passed_corr  passed_vif  passed_all  score  
0       1.0         True        True        True    4.0  
1       1.0         True        True        True    4.0  
2       1.0         True        True        True    4.0  

--- Domain: macro ---
Top options:
  domain     features  n_features  max_abs_corr  mean_abs_corr  max_vif  \
0  macro  (feature6,)           1           0.0            0.0      1.0   
1  macro  (feature7,)           1           0.0            0.0      1.0   
2  macro  (feature8,)           1           0.0            0.0      1.0   
3  macro  (feature9,)           1 

In [30]:
combined_stage1_results.head(10)

,domain,features,n_features,max_abs_corr,mean_abs_corr,max_vif,mean_vif,passed_corr,passed_vif,passed_all,score
0,menu,"(feature13, feature14)",2,0.261982,0.261982,1.074658,1.074658,True,True,True,7.877414
1,operations,"(feature22, feature25)",2,0.277021,0.277021,1.082947,1.082947,True,True,True,7.808969
2,csat,"(feature27,)",1,0.000000,0.000000,1.000000,1.000000,True,True,True,4.000000
3,csat,"(feature28,)",1,0.000000,0.000000,1.000000,1.000000,True,True,True,4.000000
4,csat,"(feature29,)",1,0.000000,0.000000,1.000000,1.000000,True,True,True,4.000000
5,macro,"(feature6,)",1,0.000000,0.000000,1.000000,1.000000,True,True,True,4.000000
6,macro,"(feature7,)",1,0.000000,0.000000,1.000000,1.000000,True,True,True,4.000000
7,macro,"(feature8,)",1,0.000000,0.000000,1.000000,1.000000,True,True,True,4.000000
8,macro,"(feature9,)",1,0.000000,0.000000,1.000000,1.000000,True,True,True,4.000000
9,media,"(feature18,)",1,0.000000,0.000000,1.000000,1.000000,True,True,True,4.000000


In [32]:
# --------------------------------------------------------
# Stage 2: Across-domain combinations
# Uses top-k domain options from stage1 to keep search manageable
# --------------------------------------------------------
stage2_results = combine_domain_options(
        df=df.drop(columns=["target"]),
        stage1_results=stage1_results,
        corr_threshold=0.30,
        vif_threshold=5.0,
        use_top_k_only_from_stage1=True,
        top_k_global=30,
        max_global_combinations=500000,
        max_checked=20,
    )

print("\n========== STAGE 2: GLOBAL TOP OPTIONS ==========")
print(stage2_results["top_k_global_options"].head(20))

# Example: inspect top 3 feature sets
if not stage2_results["top_k_global_options"].empty:
    print("\nTop 3 global feature sets:")
    top3_global_table = stage2_results["top_k_global_options"].head(3).copy()

    # Table view (similar to combined_stage1_results)
    display(top3_global_table[[
        "features",
        "n_features",
        "max_abs_corr",
        "mean_abs_corr",
        "max_vif",
        "mean_vif",
        "score"
    ]])

    # Detailed line-by-line view
    for i, row in top3_global_table.iterrows():
        print(f"\nRank {i+1}")
        print("Features:", list(row["features"]))
        print("Domain subsets:", row["domain_subsets"])
        print("n_features:", row["n_features"])
        print("max_abs_corr:", row["max_abs_corr"])
        print("max_vif:", row["max_vif"])
        print("score:", row["score"])


Potential global combinations to test: 40,320
Checked global combinations: 21
Valid global combinations: 20

========== STAGE 2: GLOBAL TOP OPTIONS ==========
                                       domain_subsets  \
0   {'csat': ['feature27'], 'macro': ['feature6'],...   
1   {'csat': ['feature27'], 'macro': ['feature6'],...   
2   {'csat': ['feature27'], 'macro': ['feature6'],...   
3   {'csat': ['feature27'], 'macro': ['feature6'],...   
4   {'csat': ['feature27'], 'macro': ['feature6'],...   
5   {'csat': ['feature27'], 'macro': ['feature6'],...   
6   {'csat': ['feature27'], 'macro': ['feature6'],...   
7   {'csat': ['feature27'], 'macro': ['feature6'],...   
8   {'csat': ['feature27'], 'macro': ['feature6'],...   
9   {'csat': ['feature27'], 'macro': ['feature6'],...   
10  {'csat': ['feature27'], 'macro': ['feature6'],...   
11  {'csat': ['feature27'], 'macro': ['feature6'],...   
12  {'csat': ['feature27'], 'macro': ['feature6'],...   
13  {'csat': ['feature27'], 'macro': ['feat

,features,n_features,max_abs_corr,mean_abs_corr,max_vif,mean_vif,score
0,"(feature27, feature6, feature18, feature13, fe...",10,0.277021,0.042336,1.097291,1.043140,48.040140
1,"(feature27, feature6, feature18, feature13, fe...",10,0.277021,0.043038,1.097851,1.043319,48.038955
2,"(feature27, feature6, feature18, feature13, fe...",10,0.277021,0.044037,1.096591,1.043767,48.038874



Rank 1
Features: ['feature27', 'feature6', 'feature18', 'feature13', 'feature14', 'feature22', 'feature25', 'feature3', 'feature10', 'feature30']
Domain subsets: {'csat': ['feature27'], 'macro': ['feature6'], 'media': ['feature18'], 'menu': ['feature13', 'feature14'], 'operations': ['feature22', 'feature25'], 'pricing': ['feature3'], 'promo': ['feature10'], 'weather': ['feature30']}
n_features: 10
max_abs_corr: 0.27702087234290534
max_vif: 1.0972912109648927
score: 48.0401402936024

Rank 2
Features: ['feature27', 'feature6', 'feature18', 'feature13', 'feature14', 'feature22', 'feature25', 'feature3', 'feature10', 'feature31']
Domain subsets: {'csat': ['feature27'], 'macro': ['feature6'], 'media': ['feature18'], 'menu': ['feature13', 'feature14'], 'operations': ['feature22', 'feature25'], 'pricing': ['feature3'], 'promo': ['feature10'], 'weather': ['feature31']}
n_features: 10
max_abs_corr: 0.27702087234290534
max_vif: 1.097851331763299
score: 48.0389546726799

Rank 3
Features: ['featu

In [34]:
stage2_results

{'all_global_options':                                        domain_subsets  \
 0   {'csat': ['feature27'], 'macro': ['feature6'],...   
 1   {'csat': ['feature27'], 'macro': ['feature6'],...   
 2   {'csat': ['feature27'], 'macro': ['feature6'],...   
 3   {'csat': ['feature27'], 'macro': ['feature6'],...   
 4   {'csat': ['feature27'], 'macro': ['feature6'],...   
 5   {'csat': ['feature27'], 'macro': ['feature6'],...   
 6   {'csat': ['feature27'], 'macro': ['feature6'],...   
 7   {'csat': ['feature27'], 'macro': ['feature6'],...   
 8   {'csat': ['feature27'], 'macro': ['feature6'],...   
 9   {'csat': ['feature27'], 'macro': ['feature6'],...   
 10  {'csat': ['feature27'], 'macro': ['feature6'],...   
 11  {'csat': ['feature27'], 'macro': ['feature6'],...   
 12  {'csat': ['feature27'], 'macro': ['feature6'],...   
 13  {'csat': ['feature27'], 'macro': ['feature6'],...   
 14  {'csat': ['feature27'], 'macro': ['feature6'],...   
 15  {'csat': ['feature27'], 'macro': ['feature6']

In [36]:
from typing import List, Tuple, Dict, Optional
import pandas as pd

def stage2_to_feature_sets(
    stage2_results: Dict[str, pd.DataFrame],
    top_k: Optional[int] = None,
    prefix: str = "s2"
) -> List[Tuple[str, Tuple[str, ...]]]:
    """
    Convert stage2_results['top_k_global_options'] or ['all_global_options']
    into the format expected by search_space_with_feature_sets:
      [(feature_set_id, (feat1, feat2, ...)), ...]
    """
    if top_k is not None:
        df_opts = stage2_results["top_k_global_options"].head(top_k).copy()
    else:
        df_opts = stage2_results["all_global_options"].copy()

    feature_sets = []
    for i, row in df_opts.reset_index(drop=True).iterrows():
        fs_id = f"{prefix}__{i+1:03d}"
        feats = tuple(row["features"])
        feature_sets.append((fs_id, feats))

    return feature_sets

In [37]:
feature_sets = stage2_to_feature_sets(
    stage2_results=stage2_results,
    top_k=20,                      # choose how many full combinations to test in parent run
    prefix="s2"
)

In [38]:
feature_sets

[('s2__001',
  ('feature27',
   'feature6',
   'feature18',
   'feature13',
   'feature14',
   'feature22',
   'feature25',
   'feature3',
   'feature10',
   'feature30')),
 ('s2__002',
  ('feature27',
   'feature6',
   'feature18',
   'feature13',
   'feature14',
   'feature22',
   'feature25',
   'feature3',
   'feature10',
   'feature31')),
 ('s2__003',
  ('feature27',
   'feature6',
   'feature18',
   'feature13',
   'feature14',
   'feature22',
   'feature25',
   'feature3',
   'feature11',
   'feature30')),
 ('s2__004',
  ('feature27',
   'feature6',
   'feature18',
   'feature13',
   'feature14',
   'feature22',
   'feature25',
   'feature3',
   'feature11',
   'feature31')),
 ('s2__005',
  ('feature27',
   'feature6',
   'feature18',
   'feature13',
   'feature14',
   'feature22',
   'feature25',
   'feature2',
   'feature10',
   'feature30')),
 ('s2__006',
  ('feature27',
   'feature6',
   'feature18',
   'feature13',
   'feature14',
   'feature22',
   'feature25',
   'feature

In [ ]:


# ============================================================
# 5) EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":
    # --------------------------------------------------------
    # Create synthetic data
    # --------------------------------------------------------
    df, feature_map = generate_synthetic_domain_data(
        n_rows=1000,
        seed=42,
        domain_feature_counts={
            "pricing": 5,
            "macro": 4,
            "promo": 2,
            "menu": 6,
            "media": 4,
            "operations": 5,
            "csat": 3,
            "weather": 2,
        }
    )

    print("\nFeature map:")
    print(feature_map.head(20))

    print("\nSynthetic dataframe shape:")
    print(df.shape)

    # --------------------------------------------------------
    # Domain-specific size restrictions
    # Example:
    # - pricing allows 1 to 3 features
    # - macro allows 1 to 2 features
    # - promo allows exactly 1 feature
    # - menu allows 1 to 3 features
    # - media allows 1 to 2 features
    # - operations allows 1 to 3 features
    # - csat allows 1 feature
    # - weather allows 1 to 2 features
    # --------------------------------------------------------
    domain_feature_restrictions = {
        "pricing": (1, 3),
        "macro": (1, 2),
        "promo": 1,
        "menu": (1, 3),
        "media": (1, 2),
        "operations": (1, 3),
        "csat": 1,
        "weather": (1, 2),
    }

    # --------------------------------------------------------
    # Stage 1: Within-domain valid subset options
    # --------------------------------------------------------
    stage1_results = generate_within_domain_options(
        df=df.drop(columns=["target"]),   # only X features for screening
        feature_map=feature_map,
        domain_feature_restrictions=domain_feature_restrictions,
        corr_threshold=0.70,
        vif_threshold=5.0,
        top_k_per_domain=8
    )

    print("\n========== STAGE 1: WITHIN-DOMAIN TOP OPTIONS ==========")
    for domain, res in stage1_results.items():
        print(f"\n--- Domain: {domain} ---")
        print("Top options:")
        print(res["top_k_options"].head(10))

    # --------------------------------------------------------
    # Stage 2: Across-domain combinations
    # Uses top-k domain options from stage1 to keep search manageable
    # --------------------------------------------------------
    stage2_results = combine_domain_options(
        df=df.drop(columns=["target"]),
        stage1_results=stage1_results,
        corr_threshold=0.70,
        vif_threshold=5.0,
        use_top_k_only_from_stage1=True,
        top_k_global=30,
        max_global_combinations=200000
    )

    print("\n========== STAGE 2: GLOBAL TOP OPTIONS ==========")
    print(stage2_results["top_k_global_options"].head(20))

    # Example: inspect top 3 feature sets
    if not stage2_results["top_k_global_options"].empty:
        print("\nTop 3 global feature sets:")
        for i, row in stage2_results["top_k_global_options"].head(3).iterrows():
            print(f"\nRank {i+1}")
            print("Features:", list(row["features"]))
            print("Domain subsets:", row["domain_subsets"])
            print("n_features:", row["n_features"])
            print("max_abs_corr:", row["max_abs_corr"])
            print("max_vif:", row["max_vif"])
            print("score:", row["score"])